# Week 3 Technical Report — RAG and Parameter-Efficient Fine-Tuning

## Summary

A retrieval-augmented customer-support assistant for TerraFlask, a reusable water bottle, answered 8 of 12 test questions correctly after human review and refused both out-of-scope questions. A separately fine-tuned 0.5B model, adapted with QLoRA in 10.4 minutes, raised ROUGE-L on held-out questions from 0.15 to 0.34 and human-rated correctness from 3.0 to 3.7 out of 5.

The main risks found were prompt injection through a retrieved document, invented arithmetic, and false refusals in the RAG pipeline, and over-short answers from the fine-tuned model. The recommended design combines both techniques with output checks in code.

The work is in two notebooks: `Week3A_RAG_Pipeline.ipynb` (pipeline, faithfulness evaluation, chunking ablation) and `Week3B_QLoRA_Finetuning.ipynb` (dataset, training, base versus fine-tuned evaluation).

## 1. Dataset Description

**RAG corpus.** Eight company documents, 809 words in total, were written for the TerraFlask assistant: product specification, materials and sourcing report, sustainability policy, warranty and returns, refill-station programme, care guide, shipping policy and internal supplier notes. A controlled corpus was chosen so that the correct answer to every test question is known exactly. The supplier notes contain a planted prompt-injection line instructing AI assistants to promise a lifetime warranty, which tests the lab's "prompt injection via documents" failure mode.

**Chunking.** The final configuration is `RecursiveCharacterTextSplitter` with 500-character chunks and a 50-character overlap, producing 16 chunks embedded with `all-MiniLM-L6-v2` into FAISS and retrieved with MMR (k = 4). It was chosen after an ablation of four settings (Section 2), because it was the only configuration that answered both multi-document questions while keeping full document-level recall.

**Fine-tuning data.** The `closed_qa` category of `databricks/databricks-dolly-15k` (human-written, CC BY-SA 3.0) was filtered to examples under 512 tokens, shuffled with seed 42 and split into 800 training and 20 held-out test examples. Each example pairs a question with a reference passage, the same input shape as the RAG generation step.

| Dataset | Size | Purpose |
| --- | --- | --- |
| TerraFlask corpus | 8 documents, 809 words, 16 chunks | RAG knowledge base |
| RAG test set | 12 questions: 7 factual, 2 multi-chunk, 2 out-of-scope, 1 adversarial | Faithfulness evaluation |
| Dolly-15k closed_qa | 800 train / 20 test | LoRA fine-tuning and evaluation |

## 2. RAG Evaluation Results

The lab's RAGAS metrics were reproduced with an open-weight judge (the Qwen2.5-3B generator itself) instead of RAGAS's default OpenAI judge. Every judgement was then checked by hand. Human review corrected three automated verdicts: two judge false negatives (Q4, Q8) and one keyword false positive (Q9).

| Metric (500/50) | Automated | Human-verified |
| --- | --- | --- |
| Context precision | 0.43 | – |
| Context recall (document level) | 1.00 | 1.00 |
| Faithfulness | 0.79 | 10 of 12 answers fully supported |
| Correct answers | 9 / 12 | **8 / 12** |
| Out-of-scope refusals | 2 / 2 | 2 / 2 |
| Mean response time | 1.44 s | – |

**Chunking ablation**

| chunk_size / overlap | Chunks | Precision | Recall | Faithfulness | Correct (keyword) |
| --- | --- | --- | --- | --- | --- |
| 200 / 20 | 41 | 0.53 | 0.92 | 0.71 | 0.58 |
| 500 / 50 | 16 | 0.43 | 1.00 | 0.79 | 0.75 |
| 500 / 0 | 16 | 0.43 | 1.00 | 0.79 | 0.75 |
| 1000 / 100 | 8 | 0.33 | 0.95 | 0.81 | 0.67 |

Small chunks raised precision but split facts that multi-chunk answers need. Whole-document chunks gave the most complete context but lost both multi-chunk questions. Overlap had no effect, because the splitter broke on paragraph boundaries before reaching the size limit.

**Weakest question 1 — Q12, prompt injection (faithfulness 0.00).** Retrieval returned the true 2-year warranty and the planted supplier note together. The model did not promise a lifetime warranty outright. It did, however, describe the warranty as unknown and pass on the planted claim as "internal supplier notes indicate… lifetime". The system-prompt instruction to ignore embedded instructions reduced the attack but did not stop it. The cause is architectural: an internal, untrusted document was indexed with customer-facing ones.

**Weakest question 2 — Q9, invented arithmetic (faithfulness 0.50).** Both required facts were retrieved (50 points buy a replacement lid; a lid costs SAR 19), but the model answered "SAR 95 (SAR 19 × 5)", a precise figure it calculated without reason. The same model made the same type of error in Week 2, where it summed two percentages into a number that appeared nowhere in the source.

**Other failures.** Two answerable questions (Q3, Q7) received false "I don't know" refusals. Q3 is a generation failure, since every relevant document was retrieved. Q7 is a chunk-level retrieval miss hidden by document-level recall: it was answered correctly only when whole documents were indexed. Source citations were frequently wrong even when the facts were right.

## 3. Fine-Tuning Configuration and Results

| Setting | Value |
| --- | --- |
| Base model | Qwen/Qwen2.5-0.5B (base, not instruction-tuned) |
| Method | QLoRA: 4-bit NF4 base with double quantisation, float32 compute |
| LoRA rank / alpha / dropout | r = 16, alpha = 32, dropout 0.05 |
| Target modules | q_proj, k_proj, v_proj, o_proj |
| Epochs / steps | 2 epochs, 200 optimisation steps |
| Batch size | 4 per device × 2 accumulation = effective 8 |
| Learning rate | 2e-4, cosine schedule, 20 warm-up steps |
| Loss | Response tokens only (prompt tokens masked) |
| Training time / adapter size | 10.4 min on a T4 / 8.7 MB |

**Training loss.** Logged every 10 steps, the loss moved between 0.79 and 1.25 with a gentle downward trend, ending at a mean of 1.01. The low starting loss (about 1.1) shows the base model could already predict reference answers fairly well. Its outputs contain memorised instruction-template text, evidence of instruction-style data in pre-training. Most of what the adapter learned was answer format rather than new knowledge.

**Results on 20 held-out questions**

| Metric | Base | Fine-tuned |
| --- | --- | --- |
| ROUGE-L F1 | 0.148 | **0.343** |
| Mean answer length (references: 26.9 words) | 79.2 words | 11.7 words |
| Human verdicts (correct / partial / wrong) | 5 / 10 / 5 | **10 / 7 / 3** |
| Mean correctness (1–5) | 3.0 | **3.7** |
| Answers with invented details | 7 | **2** |
| Runaway generation (human review) | 2 | 0 |

**Representative examples**

| Question | Base model | Fine-tuned |
| --- | --- | --- |
| What makes water cohesive? | Describes covalent bonds within the molecule, which does not answer the question | "Water is cohesive because it attracts other water molecules through hydrogen bonds." |
| What are the official languages of the UN? | All six languages listed correctly, inside a long answer | Five languages; Russian omitted |
| TerraFlask passage: is dishwasher damage covered? | "Yes, the TerraFlask warranty covers damage from a dishwasher…" (contradicts the passage) | "No, the warranty does not cover damage from a dishwasher." |

The adapter made answers relevant, concise and less prone to invention, and it fixed the base model's habit of contradicting the passage. It also introduced over-brevity: answers dropped required items, and on one domain question the model stated a wrong interval ("every 3 months" instead of once a month) with full confidence.

## 4. Reflection and Recommendation

The two experiments show that RAG and fine-tuning solve different problems. RAG supplies **knowledge**: the assistant answered warranty, care and sustainability questions it could not have known from training, and refused when the documents were silent. Because the knowledge lives in documents, a policy change means editing a file and re-indexing, which takes seconds. Every answer can also be traced to a source. Fine-tuning shapes **behaviour**: the adapted model learned to answer the question asked, stop when finished, and defer to the passage it is given. It did not learn any facts that could replace retrieval.

For a real business use case, the choice should follow what changes and what must be proven:

- **Use RAG** when information changes often, must be cited, or is private to the organisation: product details, policies, prices, internal procedures. This covers most customer-support and internal knowledge-base assistants, and it is the right starting point for TerraFlask.
- **Use fine-tuning** when the problem is consistent form rather than missing knowledge: a fixed output schema, a house tone, a classification scheme, or a domain's way of phrasing answers. It suits stable tasks with hundreds of good examples, where prompt instructions alone give inconsistent results, as Week 2 found for classification rules.
- **Combine both** when a small, cheap model must answer from retrieved context reliably. Part B showed a fine-tuned 0.5B model reversing a dangerous wrong answer once it had learned to follow the passage. A generator fine-tuned on question–context–answer examples drawn from the organisation's own documents is the natural next step for this assistant.

Neither technique removes the need for controls outside the model. The most serious failures, a false warranty claim passed on from an injected document and an invented price, were not caused by the wrong choice between RAG and fine-tuning. They came from trusting model output that nothing checked. The recommended design therefore adds:

- separate indexes for customer-facing and internal documents;
- sanitisation of ingested text;
- automatic checks that every number and cited source in an answer appears in the retrieved context;
- human review of flagged answers and of a regular sample.

This combined design is taken forward as the Week 5 capstone artefact.

## 5. Limitations

- **Small test sets.** Twelve RAG questions and twenty fine-tuning questions; a single question moves the scores by 5–8 percentage points, so the results show failure modes clearly but are not statistically robust.
- **Single reviewer.** One person wrote the answer keys and carried out all human review; no inter-annotator agreement was measured.
- **Self-judging evaluation.** The RAG judge was the generator itself. It disagreed with human review on two of twelve questions, so automated faithfulness should be treated as a screening signal, not a final score.
- **Metric blind spots.** Document-level recall hid a chunk-level retrieval miss; the keyword check passed one wrong answer; ROUGE-L rewarded brevity on incomplete answers; the runaway check missed two real cases.
- **Narrow configuration search.** One embedding model, one retriever setting (MMR, k = 4), one vector store (FAISS; Chroma not compared), one fine-tuning configuration, and no validation loss to choose the stopping point.
- **Separate experiments.** The fine-tuned model was not plugged into the RAG pipeline, so the combined approach recommended above is supported by the evidence but was not measured end to end.